# CrewAI Flow – Moltbook Agent

This notebook consolidates the full CrewFlow pipeline for interacting with **Moltbook**.

It covers:
1. **Environment setup** – imports, encoding fix, and loading `.env` variables.
2. **PostToMoltbookTool** – tool for creating posts via the Moltbook API.
3. **FetchMoltbookFeedTool** – tool for fetching the feed.
4. **UpvoteMoltbookPostTool** – tool for upvoting a post.
5. **MoltbookFlow** – the Flow class that reads a task file and dispatches an agent.
6. **Execution** – kicking off the flow.

---
## 1. Environment Setup & Imports

In [3]:
import os
import sys
import time
import requests
from typing import Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from crewai.tools import BaseTool
from crewai.flow.flow import Flow, start, listen
from crewai import Agent, Task, Crew, LLM

# Fix Windows console encoding (cp1252 can't handle emoji)
# os.environ["PYTHONIOENCODING"] = "utf-8"
# if sys.stdout.encoding != "utf-8":
#     sys.stdout.reconfigure(encoding="utf-8")
#     sys.stderr.reconfigure(encoding="utf-8")

load_dotenv()

# ── API & LLM configuration ──
API_KEY = os.getenv("MOLTBOOK_API_KEY")
BASE = "https://www.moltbook.com/api/v1"

HEARTBEAT_INTERVAL = 190  # How often to check (in seconds)
LLM_API_KEY = os.getenv("GROQ_API_KEY")
MODEL = "groq/llama-3.3-70b-versatile"

print("Environment loaded ✓")

Environment loaded ✓


---
## 2. Tool – Post to Moltbook

Creates a new post on Moltbook by sending a `POST` request to `/posts`.

In [4]:
class PostToMoltbookTool(BaseTool):
    name: str = "post_to_moltbook"
    description: str = (
        "Use this tool to create a new post on Moltbook. "
        "Provide title and content. Submolt is optional (default: general)."
    )

    def _run(self, title: str, content: str, submolt: str = "general") -> str:
        headers = {
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json"
        }
        payload = {"submolt": submolt, "title": title, "content": content}

        response = requests.post(f"{BASE}/posts", json=payload, headers=headers)

        if response.status_code in (200, 201):
            data = response.json()
            post_id = data.get("id") or data.get("post", {}).get("id") or "unknown"
            return f"Post created successfully! ID: {post_id}"
        else:
            return f"Failed: {response.status_code} - {response.text}"

print("PostToMoltbookTool defined ✓")

PostToMoltbookTool defined ✓


---
## 3. Tool – Fetch Moltbook Feed

Fetches the feed from Moltbook with configurable `sort` and `limit` parameters.  
Uses a Pydantic schema (`FetchMoltbookFeedInput`) to validate inputs.

In [5]:
class FetchMoltbookFeedInput(BaseModel):
    sort: Optional[str] = Field(
        default="hot",
        description="Sort order for the feed, e.g. 'hot', 'new', 'top'."
    )
    limit: Optional[int] = Field(
        default=5,
        description="Number of posts to return."
    )


class FetchMoltbookFeedTool(BaseTool):
    name: str = "fetch_moltbook_feed"
    description: str = (
        "Fetches the Moltbook feed and returns a list of posts. "
        "Use this to browse posts before upvoting one. "
        "Parameters sort and limit are optional with defaults 'hot' and 5."
    )
    args_schema: type = FetchMoltbookFeedInput

    def _run(self, sort: str = "hot", limit: int = 5) -> str:
        headers = {
            "Authorization": f"Bearer {API_KEY}",
        }
        params = {"sort": sort, "limit": limit}

        response = requests.get(f"{BASE}/feed", headers=headers, params=params)

        if response.status_code == 200:
            data = response.json()
            posts = data if isinstance(data, list) else data.get("posts", data.get("data", []))

            if not posts:
                return "No posts found in the feed."

            lines = []
            for i, post in enumerate(posts, 1):
                post_id = post.get("id") or post.get("_id") or "unknown"
                title = post.get("title", "No title")
                score = post.get("score", post.get("votes", "?"))
                author = post.get("author", post.get("username", "unknown"))
                if isinstance(author, dict):
                    author = author.get("username", "unknown")
                lines.append(f'{i}. [ID: {post_id}] "{title}" (score: {score}, by: {author})')

            return "Feed posts:\n" + "\n".join(lines)
        else:
            return f"Failed to fetch feed: {response.status_code} - {response.text}"

print("FetchMoltbookFeedTool defined ✓")

FetchMoltbookFeedTool defined ✓


---
## 4. Tool – Upvote a Moltbook Post

Sends a `POST` request to upvote a specific post by its `post_id`.

In [6]:
class UpvoteMoltbookPostTool(BaseTool):
    name: str = "upvote_moltbook_post"
    description: str = (
        "Upvotes a specific post on Moltbook. "
        "You must provide the post_id of the post to upvote."
    )

    def _run(self, post_id: str) -> str:
        headers = {
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        }

        response = requests.post(f"{BASE}/posts/{post_id}/upvote", headers=headers)

        if response.status_code in (200, 201):
            return f"Successfully upvoted post {post_id}!"
        else:
            return f"Failed to upvote: {response.status_code} - {response.text}"

print("UpvoteMoltbookPostTool defined ✓")

UpvoteMoltbookPostTool defined ✓


---
## 5. MoltbookFlow – The CrewAI Flow

This is the main orchestrator. It:
1. **`interpret_task_file`** – reads `tasks.md` and checks if there is pending work.
2. **`execute_agent_task`** – spins up a CrewAI Agent with the three tools above, creates a Task from the instruction, and runs the Crew.

In [7]:
class MoltbookFlow(Flow):

    @start()
    def interpret_task_file(self):
        """Read the .md file and check if there is pending work."""
        file_path = "tasks.md"
        if not os.path.exists(file_path):
            return "SLEEP"

        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read().strip()

        # If the file is empty or already marked as 'PROCESSED'
        if not content or "PROCESSED" in content:
            return "SLEEP"

        print(f"--- New instruction detected: {content} ---")
        return content

    @listen(interpret_task_file)
    def execute_agent_task(self, instruction):
        if instruction == "SLEEP":
            return "No task to execute."

        # The agent receives the instruction as-is from the file
        agent = Agent(
            role="Moltbook Proactive Agent",
            goal="Interpret instructions and interact with Moltbook using ONLY the tools provided to you.",
            backstory=(
                "You are an autonomous agent. You can create posts, fetch the feed, "
                "and upvote posts. You MUST only use the tools provided to you. "
                "Do NOT attempt to search the web or call any tool not explicitly listed."
            ),
            tools=[PostToMoltbookTool(), FetchMoltbookFeedTool(), UpvoteMoltbookPostTool()],
            llm=LLM(model=MODEL, api_key=LLM_API_KEY)
        )

        task = Task(
            description=(
                f"Follow this instruction: '{instruction}'. "
                "Use ONLY the tools provided to you. Do NOT search the web. "
                "If the instruction is about upvoting, first use fetch_moltbook_feed to browse posts, "
                "then pick one and use upvote_moltbook_post with its ID. "
                "If the instruction is about creating a post, pick an interesting topic yourself "
                "and use post_to_moltbook with submolt='general'."
            ),
            agent=agent,
            expected_output="Confirmation of the action taken."
        )

        crew = Crew(agents=[agent], tasks=[task], verbose=True)
        result = crew.kickoff()

        print("\n========== AGENT RESULT ==========")
        print(result)
        print("==================================\n")

        # Mark file as processed
        with open("tasks.md", "a", encoding="utf-8") as f:
            f.write("\n\n")

        return result

print("MoltbookFlow defined ✓")

MoltbookFlow defined ✓


---
## 6. Run the Flow

The cell below kicks off **one** iteration of the flow (reads `tasks.md` → executes the agent if work is pending).  
To run it in a continuous loop uncomment the `while True` block. 

NOTE: The LLM can sometimes hallucinate tools, which causes an error during execution. 

In [12]:
import nest_asyncio

# Jupyter and .kickoff run two different asycio events, this method helps nest them (Avoids runtime error)
nest_asyncio.apply()

# ── Single run ──
MoltbookFlow().kickoff()

Flow started with ID: 03d42a4e-6c3e-4e13-8d10-8f32e23034b3

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name:                                                                                                          │
│  MoltbookFlow                                                                                                   │
│  ID:                                                                                                            │
│  03d42a4e-6c3e-4e13-8d10-8f32e23034b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

--- New instruction detected: # upvote a post on moltbook 
# create a post on moltbook ---


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: interpret_task_file                                                                                    │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: MoltbookFlow                                                                                             │
│  ID: 03d42a4e-6c3e-4e13-8d10-8f32e23034b3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f7d0e550-5672-4155-a2fe-d18360a97c59                                                                           │
│  Final Output: Post created successfully! ID: 5840646e-a6fd-4348-adc4-598d5b5c852d                              │
│   Successfully upvoted post 27f12379-edbb-4e90-a564-317ae3c34a5d!                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: execute_agent_task                                                                                     │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  MoltbookFlow                                                                                                   │
│  ID:                                                                                                            │
│  03d42a4e-6c3e-4e13-8d10-8f32e23034b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewOutput(raw='Post created successfully! ID: 5840646e-a6fd-4348-adc4-598d5b5c852d \n Successfully upvoted post 27f12379-edbb-4e90-a564-317ae3c34a5d!', pydantic=None, json_dict=None, tasks_output=[TaskOutput(description="Follow this instruction: '# upvote a post on moltbook \n# create a post on moltbook'. Use ONLY the tools provided to you. Do NOT search the web. If the instruction is about upvoting, first use fetch_moltbook_feed to browse posts, then pick one and use upvote_moltbook_post with its ID. If the instruction is about creating a post, pick an interesting topic yourself and use post_to_moltbook with submolt='general'.", name="Follow this instruction: '# upvote a post on moltbook \n# create a post on moltbook'. Use ONLY the tools provided to you. Do NOT search the web. If the instruction is about upvoting, first use fetch_moltbook_feed to browse posts, then pick one and use upvote_moltbook_post with its ID. If the instruction is about creating a post, pick an interesting topi

In [ ]:
# ── Continuous loop (uncomment to use) ──
# while True:
#     MoltbookFlow().kickoff()
#     time.sleep(HEARTBEAT_INTERVAL)

Flow started with ID: f243a1ec-aa01-4acb-a238-f329bb69b3a2

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name:                                                                                                          │
│  MoltbookFlow                                                                                                   │
│  ID:                                                                                                            │
│  f243a1ec-aa01-4acb-a238-f329bb69b3a2                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

--- New instruction detected: # upvote a post on moltbook 
# create a post on moltbook ---


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: interpret_task_file                                                                                    │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: interpret_task_file                                                                                    │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...


========== AGENT RESULT ==========
Post created successfully! ID: 76328800-a61e-4482-8cbc-18b38f67a77c 
Successfully upvoted post 27f12379-edbb-4e90-a564-317ae3c34a5d!



╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  0981e8f1-cbd4-41ae-81c4-678d1feec883                                                                           │
│  Final Output: Post created successfully! ID: 76328800-a61e-4482-8cbc-18b38f67a77c                              │
│  Successfully upvoted post 27f12379-edbb-4e90-a564-317ae3c34a5d!                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: execute_agent_task                                                                                     │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  MoltbookFlow                                                                                                   │
│  ID:                                                                                                            │
│  f243a1ec-aa01-4acb-a238-f329bb69b3a2                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯